# Applied ML Workshop — Pneumonia Detection

**Duration:** ~2 hours | **Workflow:** Problem → Data → Explore → Preprocess → Model → Train → Validate → Test → Report

This notebook follows [`workshop_plan.md`](workshop_plan.md). Run cells top to bottom.

**Before you start:**
1. Download the [Chest X-Ray Pneumonia dataset](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia).
2. Set `DATA_ROOT` in the configuration cell below.
3. For deeper theory (SGD, linear regression), see [`1_overview.ipynb`](1_overview.ipynb).


---
## Block 1 — ML workflow & problem framing (~10 min)

*Discussion: read this section before running code.*

| Step | This notebook |
|------|---------------|
| **Define problem** | Binary classification: pneumonia vs normal on chest X-ray |
| **Choose metric** | Accuracy, precision/recall per class (false negatives may be clinically costly) |
| **Prepare data** | Load JPEGs, resize, normalize, augment, batch with `DataLoader` |
| **Choose model** | CNN (`PneumoniaCNN`) — images need spatial feature learning |
| **Train** | Minimize `BCEWithLogitsLoss` with `RMSprop` over epochs |
| **Validate** | Monitor val loss/accuracy each epoch; scheduler adjusts learning rate |
| **Test** | Final metrics on held-out test set — **once**, at the end |
| **Report** | Plots, confusion matrix, inspect misclassified images |

### Essential concepts

- **Supervised learning** — learn from labeled examples (`PNEUMONIA` / `NORMAL` folders).
- **Model** — parameterized function (CNN weights).
- **Loss** — scalar to minimize; wrong predictions increase loss.
- **Optimizer** — updates weights using gradients from the loss.
- **Epoch** — one full pass through the training set.
- **Train / val / test** — train to learn; val to monitor and tune; test to estimate real-world performance.

> **Checkpoint:** What is the real-world goal, and what proxy metric does this notebook use?


---
## Block 2 — Load & explore data (~20 min)

Set paths and load the chest X-ray dataset. Inspect class balance and sample images.


In [ ]:
# --- Configuration ---
# Path to dataset root (must contain train/, val/, test/ subfolders)
DATA_ROOT = '/path/to/chest_xray'

# Workshop demo: limit images per class for faster runs (set None for full dataset)
MAX_SAMPLES_PER_CLASS = 50

# Training epochs (use 4–6 for a 2-hour session on CPU; 12 for full run)
NUM_EPOCHS = 6

BATCH_SIZE = 32
IMG_SIZE = 150
LABELS = ['PNEUMONIA', 'NORMAL']


In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


### Dataset background

The dataset contains pediatric chest X-rays in `train/`, `val/`, and `test/` folders, each with `PNEUMONIA` and `NORMAL` subfolders. Images were quality-controlled and labeled by expert physicians before being released for ML training.


In [ ]:
def get_training_data(data_dir, max_per_class=MAX_SAMPLES_PER_CLASS):
    """Load images as (array, label) pairs. Labels: 0=PNEUMONIA, 1=NORMAL."""
    data = []
    for label in LABELS:
        path = os.path.join(data_dir, label)
        class_num = LABELS.index(label)
        filenames = sorted(os.listdir(path))
        if max_per_class is not None:
            filenames = filenames[:max_per_class]
        for img_name in filenames:
            img_path = os.path.join(path, img_name)
            img_arr = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img_arr is None:
                continue
            resized = cv2.resize(img_arr, (IMG_SIZE, IMG_SIZE))
            data.append([resized, class_num])
    return np.array(data, dtype=object)


train = get_training_data(os.path.join(DATA_ROOT, 'train'))
val = get_training_data(os.path.join(DATA_ROOT, 'val'))
test = get_training_data(os.path.join(DATA_ROOT, 'test'))

print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')


In [ ]:
# Class distribution in training set (note imbalance)
class_names = ['Pneumonia' if item[1] == 0 else 'Normal' for item in train]

sns.set_style('darkgrid')
sns.countplot(x=class_names)
plt.title('Training set class counts')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()


In [ ]:
# Preview one image from each class
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, idx in zip(axes, [0, -1]):
    ax.imshow(train[idx][0], cmap='gray')
    ax.set_title(LABELS[train[idx][1]])
    ax.axis('off')
plt.tight_layout()
plt.show()


> **Checkpoint:** Why do we keep a separate test set until the final evaluation?


---
## Block 3 — Preprocess & augment (~20 min)

Convert images to normalized tensors and build PyTorch `DataLoader`s. Apply augmentation on **training data only**.


In [ ]:
def unpack_split(data):
    x, y = [], []
    for feature, label in data:
        x.append(feature)
        y.append(label)
    return np.array(x, dtype=np.float32), np.array(y, dtype=np.float32)


x_train, y_train = unpack_split(train)
x_val, y_val = unpack_split(val)
x_test, y_test = unpack_split(test)

# Scale pixel values to [0, 1] for stable training
x_train /= 255.0
x_val /= 255.0
x_test /= 255.0

print(f'x_train shape: {x_train.shape}, pixel range: [{x_train.min():.2f}, {x_train.max():.2f}]')


In [ ]:
class PneumoniaDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).unsqueeze(0)
        return image, torch.tensor(label, dtype=torch.float32)


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

train_dataset = PneumoniaDataset(x_train, y_train, transform=train_transform)
val_dataset = PneumoniaDataset(x_val, y_val, transform=eval_transform)
test_dataset = PneumoniaDataset(x_test, y_test, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


| Technique | Why |
|-----------|-----|
| Normalize / 255 | Stable gradients; faster CNN convergence |
| Augmentation (train only) | Expand effective dataset; reduce overfitting |
| `Dataset` + `DataLoader` | Batched, shuffled input to the training loop |
| `shuffle=True` (train) | Different mini-batches each epoch |

> **Checkpoint:** Why do we divide by 255? Why not augment the test set?


---
## Block 4 — Build, train & test (~45 min)

Define the CNN, loss, and optimizer; run the training loop; evaluate on the test set.

**One training batch:**
1. Forward pass → model outputs
2. Compute loss → compare outputs to labels
3. Backward pass → gradients via autograd
4. `optimizer.step()` → update weights


In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.Dropout(0.1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.Dropout(0.2),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.Dropout(0.2),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x).squeeze(1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PneumoniaCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.RMSprop(model.parameters())
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=2, min_lr=1e-6)

print(model)
print(f'Using device: {device}')


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    running_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        if is_train:
            optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        if is_train:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == targets).sum().item()
        total += images.size(0)

    return running_loss / total, correct / total


history = {'accuracy': [], 'loss': [], 'val_accuracy': [], 'val_loss': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)
    scheduler.step(val_acc)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(
        f'Epoch {epoch + 1}/{NUM_EPOCHS} - '
        f'loss: {train_loss:.4f} - acc: {train_acc:.4f} - '
        f'val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}'
    )


In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc * 100:.2f}%')


> **Checkpoint:** Name the four steps inside one training batch.


---
## Block 5 — Evaluate & wrap-up (~25 min)

Plot learning curves, report per-class metrics, and inspect prediction errors.


In [ ]:
epochs = list(range(1, NUM_EPOCHS + 1))
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(epochs, history['accuracy'], 'go-', label='Training')
ax[0].plot(epochs, history['val_accuracy'], 'ro-', label='Validation')
ax[0].set_title('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(epochs, history['loss'], 'g-o', label='Training')
ax[1].plot(epochs, history['val_loss'], 'r-o', label='Validation')
ax[1].set_title('Loss')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
model.eval()
all_preds = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = (torch.sigmoid(outputs) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)

predictions = np.array(all_preds)
print(classification_report(
    y_test, predictions,
    target_names=['Pneumonia (Class 0)', 'Normal (Class 1)'],
))


In [ ]:
cm = confusion_matrix(y_test, predictions)
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, cmap='Blues', annot=True, fmt='d', linewidths=1, linecolor='black')
plt.title('Confusion matrix (test set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
def plot_examples(indices, title):
    n = min(6, len(indices))
    if n == 0:
        print(f'No examples for: {title}')
        return
    fig, axes = plt.subplots(2, 3, figsize=(9, 6))
    fig.suptitle(title)
    for ax, idx in zip(axes.flat, indices[:n]):
        ax.imshow(x_test[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
        ax.set_title(f'Pred {predictions[idx]}, Actual {int(y_test[idx])}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()


correct = np.where(predictions == y_test)[0]
incorrect = np.where(predictions != y_test)[0]

plot_examples(correct, 'Correctly classified')
plot_examples(incorrect, 'Incorrectly classified')


### Model selection checklist

1. Define problem and metric first.
2. Start with a clear baseline.
3. Respect train / val / test splits.
4. Monitor train vs val gap for overfitting.
5. Report test metrics and show failure cases.

### Weather data  
Size: 1000*4000 or even bigger
Channels: Instead of 3 channels for RBG like usual image data, it may have hundreds to thousands of channels. Each channel represents a 2d field, e.g. temperature, humidity etc. 


### Take-home

- [`1_overview.ipynb`](1_overview.ipynb) — linear regression, SGD, learning rate, batch size.
- [`2_neural_networks.ipynb`](2_neural_networks.ipynb) — full-length version of this workflow.
- [`bonus-finetuning.ipynb`](bonus-finetuning.ipynb) — transfer learning with ResNet-18.

> **Closing:** Name one workflow step you will reuse in your own project.
